# Test nhanh mô hình thật cho Emotion Chatbot (Colab T4)

Notebook này bật **model server** với CafeBERT (cảm xúc), PhoWhisper + librosa (giọng nói) và Qwen3-4B (câu trả lời), rồi mở tunnel **ngrok** để web thật https://emotion.hiennguyen.io.vn gọi tới.

- Module nào trong repo **đã có code thật** (Người 1, Người 3 đã nộp) thì dùng code đó; module nào còn là khung thì dùng **bản tham khảo** trong `colab/reference_modules/`. Bản tham khảo chỉ được chép vào máy Colab này, không sửa repo.
- Thời gian: khoảng 10–15 phút lần đầu (tải khoảng 12 GB mô hình).

**Chuẩn bị một lần**: bấm biểu tượng chìa khoá *Secrets* ở thanh bên trái, thêm 3 mục và bật *Notebook access*:

| Tên | Lấy ở đâu |
|---|---|
| `NGROK_AUTHTOKEN` | https://dashboard.ngrok.com/get-started/your-authtoken |
| `NGROK_DOMAIN` | https://dashboard.ngrok.com/domains (dạng `xxx.ngrok-free.app`) |
| `MODEL_SERVER_TOKEN` | chuỗi bí mật tự tạo (ô bước 0), **phải trùng** với GitHub Secret cùng tên |

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từng ô (hoặc *Runtime → Run all* khi đã có đủ Secrets).

## Bước 0 (chỉ làm một lần): tạo `MODEL_SERVER_TOKEN`
Chạy ô dưới, copy chuỗi in ra vào **Colab Secret** `MODEL_SERVER_TOKEN` **và** GitHub → Settings → Environments → production → Secret `MODEL_SERVER_TOKEN`. Sau đó xoá output của ô này (chuột phải vào output → Clear output). Khi đã có Secret thì ô này tự bỏ qua, nên *Run all* vẫn an toàn.

In [ ]:
import secrets
from google.colab import userdata
try:
    userdata.get("MODEL_SERVER_TOKEN")
    print("Đã có Secret MODEL_SERVER_TOKEN, bỏ qua bước này.")
except Exception:
    print("Token mới (copy vào Colab Secret và GitHub Secret):", secrets.token_urlsafe(24))

## Bước 1: kiểm tra GPU và đọc Secrets
Không có GPU (Colab hết quota T4) vẫn chạy được ở **chế độ CPU**: CafeBERT và PhoWhisper chạy CPU, còn câu trả lời giữ **Gemini** (Qwen3-4B quá nặng cho CPU).

In [ ]:
import os, subprocess, torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB")
    # Đúng phương án trong báo cáo: Qwen3-4B trên T4 (module llm đọc biến này).
    os.environ.setdefault("LLM_MODEL_ID", "Qwen/Qwen3-4B")
    print("LLM:", os.environ["LLM_MODEL_ID"])
else:
    print("KHÔNG CÓ GPU -> chạy chế độ CPU: CafeBERT + PhoWhisper, câu trả lời giữ Gemini.")
    print("(Muốn đủ cả Qwen: Runtime > Change runtime type > T4 GPU khi Colab cấp lại.)")

from google.colab import userdata
for key in ("NGROK_AUTHTOKEN", "NGROK_DOMAIN", "MODEL_SERVER_TOKEN"):
    try:
        os.environ[key] = userdata.get(key).strip()
    except Exception:
        raise SystemExit(f"Thiếu Secret {key}: bấm biểu tượng chìa khoá bên trái, thêm {key} và bật 'Notebook access'.")
os.environ["NGROK_DOMAIN"] = os.environ["NGROK_DOMAIN"].replace("https://", "").strip("/")
print("Đã đọc 3 Secrets (không in giá trị bí mật). Domain:", os.environ["NGROK_DOMAIN"])

## Bước 1b: lưu mô hình vào Google Drive (khuyên dùng)
Lần đầu mô hình được tải từ Hugging Face rồi **lưu vào Drive** (khoảng 12 GB, thư mục `DRIVE_CACHE` bên dưới); các lần sau đọc thẳng từ Drive, không tải lại. Colab sẽ hỏi quyền truy cập Drive: chọn tài khoản và bấm *Allow*. Drive của bạn sắp xếp khác thì sửa `DRIVE_CACHE` (đường dẫn bắt đầu bằng `/content/drive/MyDrive/`); không muốn dùng Drive thì đổi `USE_DRIVE = False`.

Ô này phải chạy **trước bước 4** (trước khi nạp mô hình).

In [ ]:
USE_DRIVE = True
# Thư mục chung của nhóm trên Drive (My Drive > Master > natural-language-processing > team)
DRIVE_CACHE = "/content/drive/MyDrive/Master/natural-language-processing/team/hf-cache"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    HF_CACHE = DRIVE_CACHE
    os.makedirs(HF_CACHE, exist_ok=True)
    os.environ["HF_HOME"] = HF_CACHE
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"  # Drive không hỗ trợ symlink, HF tự chép file
    used = subprocess.run(["du", "-sh", HF_CACHE], capture_output=True, text=True).stdout.split()
    print("Mô hình lưu ở:", HF_CACHE, "| đang có:", used[0] if used else "0")
else:
    print("Không dùng Drive: mô hình tải vào máy Colab và mất khi hết phiên.")

## Bước 1c: kiểm tra cache mô hình (tự sửa file hỏng)
Nếu phiên Colab bị ngắt giữa lúc tải, file trọng số trên Drive có thể bị cắt cụt; lần sau Hugging Face thấy file đã có nên không tải lại và PyTorch báo lỗi như `IndexError: pop from empty list`. Ô này đọc thử mọi file `.safetensors` (đọc header) và `.bin` (nạp lên CPU, `weights_only=True`) trong `HF_HOME/hub/models--*`, **xoá cả thư mục mô hình có file lỗi** để bước 4 tải lại, rồi in tên mô hình đã xoá (hoặc "Cache ổn").

Ô này phải chạy **sau bước 1b** và trước bước 4; mất thêm khoảng 10–40 giây.

In [ ]:
import gc, shutil
from pathlib import Path

# Các kiểu lỗi điển hình khi file trọng số bị cắt cụt/hỏng (dùng lại ở bước 4).
LOI_FILE_HONG = ("IndexError", "EOFError", "UnpicklingError", "OSError",
                 "SafetensorError", "HeaderTooLarge")
# RuntimeError chỉ tính là file hỏng khi đúng thông điệp đọc file .bin lỗi; không bắt lỗi
# hết VRAM (CUDA out of memory), nếu không sẽ xoá nhầm cache mô hình lớn.
_DAU_HIEU_RUNTIME = ("PytorchStreamReader", "zip archive", "unexpected EOF")

def la_loi_file_hong(e):
    ten = type(e).__name__
    if ten in LOI_FILE_HONG:
        return True
    return ten == "RuntimeError" and any(k in str(e) for k in _DAU_HIEU_RUNTIME)

def thu_muc_hub():
    """Thư mục hub của Hugging Face: theo HF_HOME (bước 1b) hoặc mặc định của Colab."""
    home = os.environ.get("HF_HOME") or os.path.join(os.path.expanduser("~"), ".cache", "huggingface")
    return Path(home) / "hub"

def xoa_cache_mo_hinh(repo_id):
    """Xoá thư mục models--<org>--<name> của repo_id; trả về True nếu có xoá."""
    d = thu_muc_hub() / ("models--" + repo_id.replace("/", "--"))
    if d.is_dir():
        shutil.rmtree(d, ignore_errors=True)
        return True
    return False

def file_hong(path):
    """Trả về chuỗi lỗi nếu file trọng số đọc không được, ngược lại trả None."""
    try:
        if path.suffix == ".safetensors":
            from safetensors import safe_open
            with safe_open(str(path), framework="pt", device="cpu"):
                pass
        else:
            import torch
            torch.load(str(path), map_location="cpu", weights_only=True)
            gc.collect()
    except Exception as e:
        return f"{type(e).__name__}: {str(e)[:100]}"
    return None

hub = thu_muc_hub()
da_xoa = []
if not hub.is_dir():
    print(f"Chưa có thư mục {hub} (chưa tải mô hình nào hoặc không dùng Drive): bỏ qua.")
else:
    for mdir in sorted(hub.glob("models--*")):
        # Chỉ kiểm file trọng số; bỏ qua training_args.bin, optimizer... (không phải state dict).
        files = sorted(set(list(mdir.rglob("*.safetensors"))
                           + list(mdir.rglob("pytorch_model*.bin"))))
        for f in files:
            loi = file_hong(f)
            if loi:
                name = mdir.name[len("models--"):].replace("--", "/")
                print(f"HỎNG: {name} / {f.name} -> {loi}")
                shutil.rmtree(mdir, ignore_errors=True)
                da_xoa.append(name)
                break
    if da_xoa:
        print("Đã xoá cache của:", ", ".join(da_xoa), "(bước 4 sẽ tải lại)")
    else:
        print("Cache ổn")

## Bước 2: tải code và cài thư viện

In [ ]:
%cd /content
!rm -rf emotion-chatbot && git clone -q --depth 1 https://github.com/NguyenTatHien/emotion-chatbot.git
%cd /content/emotion-chatbot
!pip install -q -r colab/requirements.txt "transformers>=4.56" accelerate librosa sentencepiece gTTS
!git log -1 --format="Commit: %h %s"

## Bước 3: chọn code thật hay bản tham khảo cho từng module

In [ ]:
from pathlib import Path
import shutil

for name in ("emotion", "speech", "llm"):
    target = Path(f"backend/modules/{name}.py")
    if "NotImplementedError" in target.read_text(encoding="utf-8"):
        shutil.copy(f"colab/reference_modules/{name}.py", target)
        print(f"{name:8} -> bản tham khảo (module trong repo còn là khung)")
    else:
        print(f"{name:8} -> code thật trong repo")

## Bước 4: nạp mô hình và chạy thử từng module
Lần đầu tải mô hình mất vài phút. Ô này in thời gian chạy và VRAM đã dùng.
Nếu một mô hình báo lỗi kiểu file hỏng (`IndexError`, `EOFError`, `UnpicklingError`, `OSError`, `SafetensorError`...), ô này **xoá cache của đúng mô hình đó rồi thử lại một lần**; lỗi khác hoặc lỗi lần hai vẫn báo ra như cũ.

In [ ]:
import sys, time, torch
sys.path.insert(0, "/content/emotion-chatbot/backend")
from modules import emotion, speech, llm

def timed(label, fn, *args, **kwargs):
    t = time.perf_counter()
    out = fn(*args, **kwargs)
    print(f"[{label}] {time.perf_counter() - t:.2f}s")
    return out

# Repo id lấy từ chính module (code thật hoặc bản tham khảo); không có thì dùng mặc định trong repo.
def _repo(mod, names, default):
    for n in names:
        v = getattr(mod, n, None)
        if isinstance(v, str) and v:
            return v
    return default

REPO_EMOTION = _repo(emotion, ("_MODEL_NAME", "MODEL_ID"), "BaoNhan/cafebert-UITVSMEC")
REPO_SPEECH = _repo(speech, ("_ASR_MODEL_NAME", "ASR_MODEL_ID"), "vinai/PhoWhisper-base")
REPO_LLM = os.environ.get("LLM_MODEL_ID") or os.environ.get("LLM_MODEL") or _repo(llm, ("_DEFAULT_MODEL", "MODEL_ID"), "Qwen/Qwen2.5-3B-Instruct")

def timed_retry(label, repo_id, fn, *args, **kwargs):
    """Như timed, nhưng nếu lỗi do file mô hình hỏng thì xoá cache repo_id và thử lại đúng một lần."""
    try:
        return timed(label, fn, *args, **kwargs)
    except Exception as e:
        if not la_loi_file_hong(e):
            raise
        print(f"[{label}] lỗi {type(e).__name__}: {str(e)[:100]}")
        if xoa_cache_mo_hinh(repo_id):
            print(f"[{label}] đã xoá cache {repo_id}, thử lại một lần...")
        else:
            print(f"[{label}] không thấy cache của {repo_id} để xoá, vẫn thử lại một lần...")
        return timed(label, fn, *args, **kwargs)

# Cảm xúc
for text in ["Dạo này mình áp lực thi cử, tối nào cũng khó ngủ",
             "Hôm nay mình được điểm cao, vui quá trời!",
             "Sao lại đối xử với mình như vậy, tức thật sự"]:
    r = timed_retry("emotion", REPO_EMOTION, emotion.predict_emotion, text)
    top = sorted(r["probs"].items(), key=lambda kv: -kv[1])[:3]
    print("   ", text, "->", r["label"], [(k, round(v, 2)) for k, v in top])

# Giọng nói: tạo câu nói mẫu bằng gTTS, đổi sang WAV 16 kHz mono giống backend
from gtts import gTTS
gTTS("Dạo này mình thấy hơi mệt và buồn, không muốn làm gì cả", lang="vi").save("/content/mau.mp3")
subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", "/content/mau.mp3",
                "-ac", "1", "-ar", "16000", "/content/mau.wav"], check=True)
print("    transcript:", timed_retry("transcribe", REPO_SPEECH, speech.transcribe, "/content/mau.wav"))
print("    prosody:", timed("prosody", speech.extract_prosody, "/content/mau.wav"))

# Câu trả lời (chỉ khi có GPU; CPU thì web dùng Gemini)
if HAS_GPU:
    emo = timed_retry("emotion", REPO_EMOTION, emotion.predict_emotion, "Dạo này mình áp lực thi cử, tối nào cũng khó ngủ")
    reply = timed_retry("llm", REPO_LLM, llm.generate_reply, "Dạo này mình áp lực thi cử, tối nào cũng khó ngủ",
                  emo, "low", [], goal="xoa_diu")
    print("    reply:", reply)
else:
    print("[llm] bỏ qua Qwen vì không có GPU (web giữ LLM_MODE=gemini)")

if HAS_GPU:
    print(f"VRAM đang dùng: {torch.cuda.memory_allocated() / 2**30:.1f} GB "
          f"(cao nhất {torch.cuda.max_memory_allocated() / 2**30:.1f} GB)")

## Bước 5: bật model server
Server chạy chung tiến trình với notebook nên mô hình chỉ nạp một lần (chạy tiến trình riêng sẽ nạp lại lần nữa và T4 hết VRAM).

In [ ]:
import threading, urllib.request, json, uvicorn
sys.path.insert(0, "/content/emotion-chatbot/colab")
import server_app  # dùng lại đúng các module đã nạp ở bước 4

server = uvicorn.Server(uvicorn.Config(server_app.app, host="0.0.0.0", port=8001, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(30):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)

## Bước 6: mở tunnel ngrok và kiểm tra từ Internet

In [ ]:
subprocess.run("curl -sSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz"
               " | tar xz -C /usr/local/bin", shell=True, check=True)
subprocess.run(["ngrok", "config", "add-authtoken", os.environ["NGROK_AUTHTOKEN"]],
               check=True, capture_output=True)
ngrok_proc = subprocess.Popen(
    ["ngrok", "http", "--url", os.environ["NGROK_DOMAIN"], "8001", "--log", "stdout"],
    stdout=open("/content/ngrok.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

public = "https://" + os.environ["NGROK_DOMAIN"]
req = urllib.request.Request(
    public + "/emotion", method="POST",
    data=json.dumps({"text": "mình lo lắng quá"}).encode(),
    headers={"Content-Type": "application/json",
             "X-Model-Server-Token": os.environ["MODEL_SERVER_TOKEN"],
             "ngrok-skip-browser-warning": "1"})
t = time.perf_counter()
answer = urllib.request.urlopen(req, timeout=30).read().decode()
print("Gọi qua Internet:", answer[:120], f"({time.perf_counter() - t:.2f}s)")
print()
print("Xong! Trên GitHub -> Settings -> Environments -> production đặt:")
print("  Variable MODEL_SERVER_URL =", public)
print("  Variable EMOTION_MODE     = remote")
print("  Variable SPEECH_MODE      = remote")
if HAS_GPU:
    print("  Variable LLM_MODE         = remote   (Qwen; Gemini tự lùi xuống làm dự phòng)")
else:
    print("  Variable LLM_MODE         = gemini   (giữ nguyên: CPU không chạy Qwen)")
print("  Secret   MODEL_SERVER_TOKEN phải trùng với Colab")
print("rồi Actions -> Deploy -> Run workflow. Mở web: header hiện 'Máy chủ mô hình'.")
print("Giữ tab này mở; Colab ngắt khi bỏ không khoảng 90 phút.")

## Theo dõi (tuỳ chọn)
Xem 20 dòng log ngrok gần nhất (mỗi lượt chat trên web sinh vài dòng).

In [ ]:
!tail -n 20 /content/ngrok.log

## Tắt (chỉ khi muốn dừng)
Ô này **không tự tắt khi bấm Run all**. Muốn dừng model server và ngrok: đổi `STOP = True` rồi chạy riêng ô này. (Hoặc *Runtime → Disconnect and delete runtime*.)
Không đặt lại Variable trên GitHub cũng không sao: web tự chạy ở chế độ dự phòng khi Colab tắt.

In [ ]:
STOP = False  # đổi thành True rồi chạy riêng ô này để tắt

if STOP:
    server.should_exit = True
    ngrok_proc.terminate()
    print("Đã tắt model server và ngrok.")
else:
    print("Model server và ngrok vẫn đang chạy (STOP = False).")

## Sửa nhanh khi web vẫn báo "Dự phòng"
Chạy ô dưới: nó kiểm tra server trong máy, tiến trình ngrok và đường từ Internet vào, rồi **tự bật lại** phần nào đang tắt (không nạp lại mô hình).

In [ ]:
import urllib.request

def _ok(url, headers=None):
    try:
        req = urllib.request.Request(url, headers=headers or {})
        return urllib.request.urlopen(req, timeout=10).status == 200
    except Exception:
        return False

local_ok = _ok("http://127.0.0.1:8001/health")
if not local_ok:
    server = uvicorn.Server(uvicorn.Config(server_app.app, host="0.0.0.0", port=8001, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    time.sleep(3)
    local_ok = _ok("http://127.0.0.1:8001/health")
print("Model server trong máy:", "OK" if local_ok else "LỖI")

ngrok_alive = "ngrok_proc" in globals() and ngrok_proc.poll() is None
public = "https://" + os.environ["NGROK_DOMAIN"]
public_ok = _ok(public + "/health", {"ngrok-skip-browser-warning": "1"})
if not (ngrok_alive and public_ok):
    subprocess.run(["pkill", "-f", "ngrok http"], check=False)
    ngrok_proc = subprocess.Popen(
        ["ngrok", "http", "--url", os.environ["NGROK_DOMAIN"], "8001", "--log", "stdout"],
        stdout=open("/content/ngrok.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    public_ok = _ok(public + "/health", {"ngrok-skip-browser-warning": "1"})
print("Tunnel ngrok từ Internet:", "OK" if public_ok else "LỖI - xem: !tail -n 30 /content/ngrok.log")